# 1) Mount drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 2) Import libraries

In [2]:
# Most basic stuff for EDA.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Core packages for text processing.

import string
import re

# Libraries for text preprocessing.

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist

#import the RegexpTokenizer library
from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer
from nltk.tokenize import regexp_tokenize

# Loading some sklearn packaces for modelling.

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.metrics import f1_score, accuracy_score
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import NMF


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


# 3) read data

In [3]:
df = pd.read_csv('/content/drive/MyDrive/ML/computational data mining/تمرین نهم/train.csv')
df

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


# 4) Preprocess data

In [4]:
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


In [6]:
df.isnull().sum()

,0
id,0
keyword,61
location,2533
text,0
target,0


In [7]:
# Some basic helper functions to clean text by removing urls, emojis, html tags and punctuations.

def remove_URL(text):
    url = re.compile(r'https?://\S+|www\.\S+')
    return url.sub(r'', text)


def remove_emoji(text):
    emoji_pattern = re.compile(
        '['
        u'\U0001F600-\U0001F64F'  # emoticons
        u'\U0001F300-\U0001F5FF'  # symbols & pictographs
        u'\U0001F680-\U0001F6FF'  # transport & map symbols
        u'\U0001F1E0-\U0001F1FF'  # flags (iOS)
        u'\U00002702-\U000027B0'
        u'\U000024C2-\U0001F251'
        ']+',
        flags=re.UNICODE)

    return emoji_pattern.sub(r'', text)

# Removing HTML tags
def remove_html(text):
    html = re.compile(r'<.*?>|&([a-z0-9]+|#[0-9]{1,6}|#x[0-9a-f]{1,6});')
    return re.sub(html, '', text)

# Removing punctuations
def remove_punct(text):
    table = str.maketrans('', '', string.punctuation)
    return text.translate(table)

# Converting to lowercase
def convert_to_lowercase(text):
    return text.lower()

# Removing whitespaces
def remove_whitespace(text):
    return text.strip()

# Function to remove stopwords from a list of texts
def remove_stopwords(text):
    # Stopwords
    stops = stopwords.words("english") # stopwords
    addstops = ["among", "onto", "shall", "thrice", "thus", "twice", "unto", "us", "would"] # additional stopwords
    allstops = stops + addstops
    # \w+ matches words and numbers, omitting punctuation completely.
    regexp = RegexpTokenizer(r'\w+')
    return " ".join([word for word in regexp.tokenize(text) if word not in allstops])


def remove_numeric_characters(text):
    return re.sub('[\d]','',text) # this will remove numeric characters


# Stemming
stemmer = PorterStemmer()
def text_stemmer(text):
    regexp = RegexpTokenizer(r'\w+')
    text_stem = " ".join([stemmer.stem(word) for word in regexp.tokenize(text)])
    return text_stem

# Applying helper functions

df['new_text'] = df['text'].apply(lambda x: remove_URL(x))
df['new_text'] = df['new_text'].apply(lambda x: remove_emoji(x))
df['new_text'] = df['new_text'].apply(lambda x: remove_html(x))
df['new_text'] = df['new_text'].apply(lambda x: remove_punct(x))
df['new_text'] = df['new_text'].apply(lambda x: convert_to_lowercase(x))
df['new_text'] = df['new_text'].apply(lambda x: remove_whitespace(x))
df['new_text'] = df['new_text'].apply(lambda x: remove_stopwords(x))
df['new_text'] = df['new_text'].apply(lambda x: remove_numeric_characters(x))
df['new_text'] = df['new_text'].apply(lambda x: text_stemmer(x))

<>:52: SyntaxWarning: invalid escape sequence '\d'
<>:52: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_1501/3255446977.py:52: SyntaxWarning: invalid escape sequence '\d'
  return re.sub('[\d]','',text) # this will remove numeric characters


In [8]:
df.head()

,id,keyword,location,text,target,new_text
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1,deed reason earthquak may allah forgiv
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1,forest fire near la rong sask canada
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1,resid ask shelter place notifi offic evacu she...
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1,peopl receiv wildfir evacu order california
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1,got sent photo rubi alaska smoke wildfir pour ...


## Stemming

In [9]:
df=df.drop(['id', 'keyword', 'location', 'text', 'target'],axis=1)
df.head()

,new_text
0,deed reason earthquak may allah forgiv
1,forest fire near la rong sask canada
2,resid ask shelter place notifi offic evacu she...
3,peopl receiv wildfir evacu order california
4,got sent photo rubi alaska smoke wildfir pour ...


In [10]:
df = df.drop_duplicates()
df

,new_text
0,deed reason earthquak may allah forgiv
1,forest fire near la rong sask canada
2,resid ask shelter place notifi offic evacu she...
3,peopl receiv wildfir evacu order california
4,got sent photo rubi alaska smoke wildfir pour ...
...,...
7601,break la refugio oil spill may costlier bigger...
7602,siren went wasnt forney tornado warn
7603,offici say quarantin place alabama home possib...
7605,flip side im walmart bomb everyon evacu stay t...


In [11]:
df.reset_index(inplace=True)
df

,index,new_text
0,0,deed reason earthquak may allah forgiv
1,1,forest fire near la rong sask canada
2,2,resid ask shelter place notifi offic evacu she...
3,3,peopl receiv wildfir evacu order california
4,4,got sent photo rubi alaska smoke wildfir pour ...
...,...,...
6855,7601,break la refugio oil spill may costlier bigger...
6856,7602,siren went wasnt forney tornado warn
6857,7603,offici say quarantin place alabama home possib...
6858,7605,flip side im walmart bomb everyon evacu stay t...


## TF-IDF Model

In order to perform machine learning on text data, we must transform the documents into vector representations. In natural language processing, text vectorization is the process of converting words, sentences, or even larger units of text data to numerical vectors.

In [12]:
# TF-IDF vectorization
TfidfVec = TfidfVectorizer(ngram_range = (1, 1))
tfidf = TfidfVec.fit_transform(df['new_text'])

In [13]:
tfidf.shape

(6860, 13729)

In [14]:
A = tfidf.transpose()
print(A.shape)

(13729, 6860)


In [15]:
print(type(A))

<class 'scipy.sparse._csc.csc_matrix'>


# 5) first algorithm (SVD)

In [16]:
# Create a TruncatedSVD object with n_components=10
svd = TruncatedSVD(n_components=10)

# Fit the TruncatedSVD model to the data
A_transformed = svd.fit_transform(A)

In [17]:
A_transformed.shape

(13729, 10)

In [18]:
svd.components_.shape

(10, 6860)

In [19]:
U1 = A_transformed[:, 0]
V1 = svd.components_[0, :]
print('shape of U1: ', U1.shape)
print('shape of V1: ', V1.shape)

shape of U1:  (13729,)
shape of V1:  (6860,)


In [20]:
print('tyep of U1: ', type(U1))
print('type of V1: ', type(V1))

tyep of U1:  <class 'numpy.ndarray'>
type of V1:  <class 'numpy.ndarray'>


In [21]:
print(U1[:10])
print(V1[:10])

[0.01080226 0.00460281 0.0018686  0.0044545  0.00820137 0.00198683
 0.0023376  0.0028889  0.00172482 0.00093108]
[0.00379727 0.01492341 0.00695296 0.01843141 0.01072387 0.01679446
 0.01411399 0.03558006 0.02003955 0.02559728]


10 most important key words

In [22]:
word_score = np.abs(U1)
sentence_score = np.abs(V1)

word_idx = np.argsort(word_score)[::-1]
sentence_idx = np.argsort(sentence_score)[::-1]

print(word_idx[:10])
print(sentence_idx[:15])

[5748 6917 4267 4744 3362 1656 4841 8580 1391 1625]
[3441 3448 2741 1088 3451 2937 2552 1225 1081 3136  886 2245  426 1139
 1571]


In [23]:
print('\nidf values:')
list_of_words = []
for ele1 in TfidfVec.get_feature_names_out():
    list_of_words.append(ele1)
print(len(list_of_words))
print(list_of_words[:3])


idf values:
13729
['aa', 'aaaa', 'aaaaaaallll']


In [24]:
key_words = [list_of_words[i] for i in word_idx[:10]]
print('key words: ', key_words)

key words:  ['im', 'like', 'fire', 'get', 'dont', 'burn', 'go', 'one', 'bomb', 'build']


In [25]:
key_sentences = df[df.index.isin(sentence_idx[:15])]
key_sentences

,index,new_text
426,486,im feel attack
886,985,mattbez oh im bag bodi bangin im say she go ge...
1081,1197,mmmmmm im burn im burn build im build oooooohh...
1088,1204,im mental prepar bomb ass school year im burn ...
1139,1257,hope time end tv im arrest light build fire
1225,1350,im battl monster im pull burn build say ill gi...
1571,1722,look back daughter said everyon love pictur po...
2245,2424,train derail instead get work earli like would...
2552,2783,im disast
2741,2989,feel like im drown insid bodi


15 most important sentences

In [26]:
key_sentences = df[df.index.isin(sentence_idx[:15])].loc[sentence_idx[:15]].reset_index()
key_sentences

,level_0,index,new_text
0,3441,3751,im fire
1,3448,3758,well feel like im fire
2,2741,2989,feel like im drown insid bodi
3,1088,1204,im mental prepar bomb ass school year im burn ...
4,3451,3761,noth like good fire
5,2937,3192,bodi like go fuck sleep sami mind like make em...
6,2552,2783,im disast
7,1225,1350,im battl monster im pull burn build say ill gi...
8,1081,1197,mmmmmm im burn im burn build im build oooooohh...
9,3136,3410,swea feel like im explod


# 6) second algorithm (NMF)

In [27]:
# NMF_model = NMF(n_components=10, random_state=1)
NMF_model = NMF(n_components=100, init='random', random_state=0)
W = NMF_model.fit_transform(A)
H = NMF_model.components_

In [28]:
print('shape of W: ', W.shape)
print('shape of H: ', H.shape)

shape of W:  (13729, 100)
shape of H:  (100, 6860)


In [29]:
H_norm = np.linalg.norm(H, axis=0)
print(H_norm[:10])
print('-' * 20)
print(H_norm.shape)
print('-' * 20)
H_norm_sorted = np.argsort(H_norm)[::-1]
print(H_norm_sorted[:15])

[0.1341692  0.13136432 0.14426955 0.3185297  0.18120252 0.17217714
 0.26404503 0.2457309  0.26941468 0.15174305]
--------------------
(6860,)
--------------------
[4741 4755 6669 6678 6654 4969 3259 4992 6662 6655 5520 6682 1431 3269
 1726]


In [30]:
key_sentences = df.loc[H_norm_sorted[:15]].reset_index()
key_sentences

,level_0,index,new_text
0,4741,5184,obliter
1,4755,5199,obliter last night
2,6669,7377,texa seek comment rule chang windstorm insur
3,6678,7388,insur texa seek comment rule chang windstorm i...
4,6654,7362,texa seek comment rule chang windstorm insur i...
5,4969,5465,reddit quarantin offens content
6,3259,3547,russian food crematoria provok outrag amid cri...
7,4992,5503,offici alabama home quarantin possibl ebola case
8,6662,7370,insur texa seek comment rule chang windstorm i...
9,6655,7363,ij texa seek comment rule chang windstorm insur


In [31]:
H.shape

(100, 6860)

In [32]:
C = W.copy()
D = H.copy()

In [33]:
import numpy as np

desired_sen = 15

# Initial order of the sentences
sentence_order = np.arange(D.shape[1])

for i in range(desired_sen):

    # 1. Find the most important column from the remaining part of D
    D_norm = np.linalg.norm(D[i:, :], axis=0)
    selected_idx = np.argmax(D_norm)

    # 2. Swap the selected column with column i
    D[:, [i, selected_idx]] = D[:, [selected_idx, i]]
    sentence_order[[i, selected_idx]] = sentence_order[[selected_idx, i]]

    # 3. Construct vector x from the selected column
    x = D[i:, i].copy()

    # 4. Construct the Householder vector
    norm_x = np.linalg.norm(x)

    if norm_x != 0:
        sign = 1 if x[0] >= 0 else -1

        v = x.copy()
        v[0] += sign * norm_x

        # 5. Construct the Householder matrix
        Q_small = np.eye(len(v)) - 2 * np.outer(v, v) / np.dot(v, v)

        # 6. Embed Q_small into a full Q matrix
        Q = np.eye(D.shape[0])
        Q[i:, i:] = Q_small

        # 7. Update C and D
        C = C @ Q
        D = Q.T @ D

# Get the original indices of the 15 selected sentences
key_sentences_idx = sentence_order[:desired_sen]

# Display the selected sentences
for idx in key_sentences_idx:
    print(df.iloc[idx]['new_text'])

obliter
texa seek comment rule chang windstorm insur
reddit quarantin offens content
russian food crematoria provok outrag amid crisi famin memori
offici alabama home quarantin possibl ebola case
fuck
hope fall cliff
crush
see fire
surviv
cameroon repatri nigerian refuge
way destroy
death
abcnew obama declar disast typhoondevast saipan obama sign disast declar northern
jax issu hazard weather outlook hwo
